In [1]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Some global variables

In [2]:
filter_start = "2025-10-01"
filter_end = "2026-10-01"

period_start = "2023-10-01"
period_end = "2026-10-01"

Extracting stocks

In [3]:
# EXTRACTING TICKERS. MAKE SURE TO CHANGE FILE NAME FOR ACTUAL SUBMISSION
tickers = pd.read_csv("Tickers.csv", header=None)[0].to_list()
close_prices = pd.DataFrame()

for tick in tickers:
    t = yf.Ticker(tick)
    information = t.info

    # Check if there is data in here (i.e. not a delisted or invalid ticker)
    if t.history(period='max').empty:
        print(tick, "doesn't exist")
        continue
    
    # Check if it's a CAD or USD stock
    if (information["market"] != "ca_market") and (information["market"] != "us_market"):
        print(tick, "wrong market")
        continue

    # Check if it's a stock and not something like a bond or index fund
    if information["quoteType"] != "EQUITY":
        print(tick, "not stock")
        continue

    if (information["currency"] != "CAD") and (information["currency"] != "USD"):
        print(tick, "currency wrong")
        continue
    

    temp = yf.download(tick, filter_start, filter_end)

    if temp["Volume"].mean()[tick] < 5000:
        print(tick, "not enough volume")
        continue
    
    close_prices[tick] = temp["Close"]

# Filters out the first and last trading day of each month
idx = close_prices.index.to_series()
date_series_first=idx.groupby([idx.dt.year, idx.dt.month]).first()
date_series_last=idx.groupby([idx.dt.year, idx.dt.month]).last()
monthly_first = []
monthly_last = []
for i in date_series_first:
    monthly_first.append(i)
for i in date_series_last:
    monthly_last.append(i)

for i in range(len(monthly_first)):
    start_ind = close_prices.index.get_loc(monthly_first[i])
    end_ind = close_prices.index.get_loc(monthly_last[i])

    if (end_ind - start_ind + 1) < 18:
        close_prices.drop(close_prices.loc[monthly_first[i]:monthly_last[i]].index)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$AGN: possibly delisted; no timezone found


AGN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$BK: possibly delisted; no price data found  (1d 1927-12-14 -> 2026-11-19)


BK doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$CELG: possibly delisted; no timezone found


CELG doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$MON: possibly delisted; no timezone found


MON doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
$RTN: possibly delisted; no timezone found


RTN doesn't exist


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
display(close_prices)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-01,254.316605,235.580231,129.656555,236.889038,77.725342,220.630005,324.080444,215.199997,49.622253,6.67,...,65.924820,163.093628,208.619995,20.261112,107.826416,174.982300,339.266235,229.894775,78.023788,46.106377
2025-10-02,255.989151,228.041824,129.190277,237.501389,77.725342,222.410004,325.974243,217.429993,49.426430,6.63,...,68.169777,165.405457,211.240005,19.976391,106.874413,176.854431,344.545624,229.679321,79.124153,46.135265
2025-10-03,256.875183,225.487259,130.744537,238.453964,79.072174,219.509995,325.737488,216.300003,49.583088,6.28,...,68.486244,165.728714,224.889999,20.104973,108.913048,174.914398,350.857574,231.912262,80.243011,46.289337
2025-10-06,255.551102,221.901199,129.918839,241.224197,81.024086,220.899994,327.463623,219.729996,49.338306,6.68,...,70.503746,165.180145,229.520004,19.893730,108.951515,176.359711,349.464630,232.509674,80.095062,45.952305
2025-10-07,255.342056,224.446152,129.219421,244.198547,82.029327,221.779999,323.498444,221.820007,49.240395,6.48,...,73.787132,162.084625,225.100006,19.875360,108.605324,171.742416,354.227814,227.074219,79.503265,45.740452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.664978,263.502045,100.451622,175.998123,74.820000,249.380005,304.717529,196.800003,56.029999,12.27,...,52.599998,194.259995,205.300003,12.030000,167.495621,269.335388,375.010010,274.070007,90.706924,57.857586
2026-09-25,340.811066,262.726807,100.670280,174.708466,74.169998,249.669998,307.937592,198.070007,56.700001,11.57,...,55.040001,201.970001,201.199997,12.000000,169.511597,276.719360,376.589996,273.790009,92.589050,58.778946
2026-09-28,338.143066,264.654968,100.362175,173.081512,72.779999,246.149994,305.385437,184.389999,55.470001,12.51,...,54.279999,187.479996,204.199997,11.860000,168.429123,276.958191,377.829987,274.100006,92.874825,58.234055


In [5]:
full_data = pd.DataFrame()

for tick in close_prices:
    full_data[tick] = yf.download(tick, period_start, period_end)["Close"]

full_data["CBIL.TO"] = yf.download("CBIL.TO", period_start, period_end, auto_adjust=True)["Close"]

full_data = full_data

display(full_data)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB,CBIL.TO
Date,,,,,,,,,,,,,,,,,,,,,
2023-10-02,171.301208,132.844528,89.382942,289.540100,56.168285,129.460007,143.918549,187.830002,24.848059,6.02,...,104.262360,73.839996,17.100689,70.040115,146.078217,483.225739,189.679672,128.719498,28.285889,45.243145
2023-10-03,169.970245,132.109756,89.438965,287.633362,54.849434,124.720001,139.653381,188.919998,24.112850,5.98,...,103.408379,70.440002,17.204720,69.378120,143.612381,478.436462,190.746765,127.615578,27.581821,45.229576
2023-10-04,171.212463,132.342773,89.308243,290.601410,55.522896,127.000000,141.174606,186.729996,24.140772,5.86,...,103.089294,72.230003,17.756868,69.195206,144.269913,479.516388,189.333328,127.590675,27.493818,45.243145
2023-10-05,172.444870,132.127670,89.821770,290.855042,56.261829,125.959999,142.199341,186.289993,24.252445,5.20,...,103.042374,71.510002,18.100965,69.997528,141.657959,484.784637,187.807617,127.308472,28.048269,45.247662
2023-10-06,174.988480,132.835587,90.456696,293.231415,56.935276,127.959999,144.355164,187.380005,24.261757,5.15,...,104.055901,73.269997,18.044949,70.359039,143.384033,492.841949,189.511215,128.047165,28.039467,45.256706
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-24,335.664978,263.502045,100.451622,175.998123,74.820000,249.380005,304.717529,196.800003,56.029999,12.27,...,194.259995,205.300003,12.030000,167.495621,269.335388,375.010010,274.070007,90.706924,57.857586,49.906170
2026-09-25,340.811066,262.726807,100.670280,174.708466,74.169998,249.669998,307.937592,198.070007,56.700001,11.57,...,201.970001,201.199997,12.000000,169.511597,276.719360,376.589996,273.790009,92.589050,58.778946,49.906170
2026-09-28,338.143066,264.654968,100.362175,173.081512,72.779999,246.149994,305.385437,184.389999,55.470001,12.51,...,187.479996,204.199997,11.860000,168.429123,276.958191,377.829987,274.100006,92.874825,58.234055,49.906170


Sharpe calculation

In [6]:
half_life = 100

In [7]:
def get_stats(tick, half_life):
    s = full_data[tick].dropna()
    delta = np.log(1 + s.pct_change()).ewm(halflife = half_life)
    standard_deviation = delta.std().iloc[-1]
    average = delta.mean().iloc[-1]

    return standard_deviation, average

In [8]:
Rf = get_stats("CBIL.TO", half_life)[0]
print("Risk free return:", Rf)

sharpe = {}

for tick in close_prices:
    stats = get_stats(tick, half_life)
    sharpe[tick] = (stats[0] - Rf)/stats[1]

sharpe = dict(sorted(sharpe.items(), key = lambda item:item[1]))
print(sharpe)

Risk free return: 0.00015446073668472015
{'PYPL': np.float64(-117.21740093444376), 'CL': np.float64(-101.15087261178155), 'AIG': np.float64(-99.65043488794079), 'LMT': np.float64(-86.07644889312118), 'ACN': np.float64(-44.41514910978841), 'AXP': np.float64(-44.101680139247954), 'UPS': np.float64(-38.994606335962686), 'ABT': np.float64(-32.822480984408855), 'BA': np.float64(-23.418144746095287), 'PEP': np.float64(-16.75904132993744), 'T.TO': np.float64(-7.562647141678668), 'TD.TO': np.float64(7.155565884406321), 'RY.TO': np.float64(8.57391877538563), 'MRK': np.float64(11.108332881807225), 'BB.TO': np.float64(11.902388082502473), 'AAPL': np.float64(13.396587662744153), 'ABBV': np.float64(14.302960827589311), 'KO': np.float64(14.541885469846134), 'BIIB': np.float64(15.875197813275868), 'PFE': np.float64(16.13762333310726), 'C': np.float64(19.943976281104103), 'TXN': np.float64(19.98388786609983), 'CAT': np.float64(20.118196748611766), 'PM': np.float64(20.444031600049623), 'BMY': np.float6

In [9]:
def weighted_corr(t1, t2):
    d1 = full_data[t1].pct_change(fill_method=None)
    d2 = full_data[t2].pct_change(fill_method=None)
    return d1.ewm(halflife=half_life).corr(d2).dropna().iloc[-1]

Correlation matrix

In [10]:
corr_mat = pd.DataFrame()

for t1 in close_prices:
    for t2 in close_prices:
        corr_mat.loc[t1, t2] = weighted_corr(t1, t2)

In [11]:
display(corr_mat)

,AAPL,ABBV,ABT,ACN,AIG,AMZN,AXP,BA,BAC,BB.TO,...,PYPL,QCOM,SHOP.TO,T.TO,TD.TO,TXN,UNH,UNP,UPS,USB
AAPL,1.000000,0.169150,0.098866,0.180679,0.229777,0.111152,0.246738,0.237700,0.251212,0.022890,...,0.269392,0.214442,0.261143,0.326863,0.223009,0.135810,-0.035881,0.108889,0.215455,0.267526
ABBV,0.169150,1.000000,0.321891,0.187360,0.194110,-0.155674,0.070052,-0.036207,0.075035,-0.015386,...,0.138632,-0.094889,0.016319,0.113246,0.022721,-0.077463,0.135933,0.199028,0.160271,0.138871
ABT,0.098866,0.321891,1.000000,0.285273,0.222522,-0.034579,0.111076,0.081138,0.135495,-0.225666,...,0.169627,-0.137684,0.157454,0.099338,0.000159,-0.144920,0.091165,0.256965,0.141354,0.194551
ACN,0.180679,0.187360,0.285273,1.000000,0.213496,0.088046,0.269786,0.095374,0.107067,0.027500,...,0.301324,-0.026642,0.385755,0.288612,-0.037426,-0.257378,0.074579,0.076987,0.120719,0.213970
AIG,0.229777,0.194110,0.222522,0.213496,1.000000,0.015158,0.185538,0.147174,0.244671,-0.100936,...,0.076436,-0.033823,0.122275,0.052264,0.147040,-0.012408,0.099625,0.288567,0.119157,0.258253
AMZN,0.111152,-0.155674,-0.034579,0.088046,0.015158,1.000000,0.322887,0.273834,0.254508,0.134281,...,0.210283,0.170858,0.226590,-0.286971,0.224261,0.133083,0.038342,0.053068,0.123004,0.255500
AXP,0.246738,0.070052,0.111076,0.269786,0.185538,0.322887,1.000000,0.383087,0.598730,0.158492,...,0.305911,0.202895,0.370172,0.081958,0.429763,0.152980,0.193191,0.180926,0.281189,0.666731
BA,0.237700,-0.036207,0.081138,0.095374,0.147174,0.273834,0.383087,1.000000,0.323247,0.045766,...,0.205021,0.276180,0.184316,0.044927,0.306190,0.263537,0.110285,0.069644,0.235342,0.320842
BAC,0.251212,0.075035,0.135495,0.107067,0.244671,0.254508,0.598730,0.323247,1.000000,0.092415,...,0.165409,0.189872,0.146564,-0.028299,0.505531,0.165842,0.115228,0.207871,0.264538,0.743015
BB.TO,0.022890,-0.015386,-0.225666,0.027500,-0.100936,0.134281,0.158492,0.045766,0.092415,1.000000,...,0.071328,0.218934,0.181671,-0.082448,0.224928,0.205908,0.045081,0.038184,0.156753,0.137061
